# Diagnostics — Channel Behavior on Cleaned Data

**Goal:** Before committing to an attribution model (Markov removal-effect, Shapley
value), sanity-check how channels actually behave in converting journeys. In
particular: does last-touch attribution (the naive default many teams use) tell a
different story than first-touch, and does any single channel dominate journeys in a
way that would make a simple model misleading?

**Input:** `touchpoints_clean_v3.csv` (post bot-removal)
**Used by:** Findings here motivate the model choices and channel handling in
`Models.ipynb`.

In [2]:
import pandas as pd

## Step 1 — Load cleaned data

Loads the bot-filtered dataset from `BOT_Detection3.ipynb` and re-applies the same
text normalization (channel casing, event-type whitespace) for consistency.

In [4]:
df = pd.read_csv('touchpoints_clean_v3.csv', parse_dates=['timestamp'])
df['channel']    = df['channel'].str.strip().str.title()
df['event_type'] = df['event_type'].str.strip()
df = df.sort_values(['user_id', 'timestamp'])

## Step 2 — Last-touch channel for every Purchase

For every conversion, which channel was the *Purchase event itself* logged on? This is
last-touch attribution by definition — whichever channel happens to host the final
click before conversion gets 100% of the credit. It's the simplest possible
attribution rule, and useful as a baseline to compare richer models against.

Google Search dominates here (48.4%), which raises the question of whether that
reflects genuine influence or just "Search tends to be where people search right
before they're ready to buy" — a timing artifact rather than a causal one.

In [8]:
# For every converting user, what channel did the Purchase event happen ON?
purchases = df[df['event_type'] == 'Purchase']
print("Which channel is the Purchase event logged on? (last-touch by definition)")
print(purchases['channel'].value_counts())
print("\nAs %:")
print((purchases['channel'].value_counts(normalize=True)*100).round(2))


Which channel is the Purchase event logged on? (last-touch by definition)
channel
Google Search      2661
Marketplace        1024
Instagram           873
Influencer Blog     501
Youtube             439
Name: count, dtype: int64

As %:
channel
Google Search      48.40
Marketplace        18.62
Instagram          15.88
Influencer Blog     9.11
Youtube             7.98
Name: proportion, dtype: float64


## Step 3 — First-touch channel for converters (top-of-funnel view)

Rebuild each user's full channel journey (ordered list of channels touched) and a
conversion flag, then check which channel started each converting journey. If
first-touch and last-touch tell very different stories, that's a sign a single-touch
attribution rule is hiding real multi-channel influence — which is exactly the
motivation for using Markov / Shapley models instead of last-click.

Note: first-touch results are far more evenly spread across channels (Google Search
26.6%, Influencer Blog 20.9%, etc.) than the last-touch numbers above — channels other
than Search are clearly doing top-of-funnel work that last-click attribution would
erase entirely.

In [11]:
# For converting users, what was their FIRST channel (top of funnel)?
def journey(g):
    return pd.Series({'path': g['channel'].tolist(), 'converted': (g['event_type']=='Purchase').any()})

journeys = df.groupby('user_id', group_keys=False).apply(journey)
converting = journeys[journeys['converted']]

print(f"\nTotal converting journeys: {len(converting)}")
print("\nFirst-touch channel (top of funnel) for converters:")
first_touch = converting['path'].apply(lambda p: p[0]).value_counts(normalize=True)*100
print(first_touch.round(2))


Total converting journeys: 5498

First-touch channel (top of funnel) for converters:
path
Google Search      26.59
Influencer Blog    20.86
Instagram          17.97
Marketplace        17.95
Youtube            16.62
Name: proportion, dtype: float64


C:\Users\mohit\AppData\Local\Temp\ipykernel_3672\4051446740.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  journeys = df.groupby('user_id', group_keys=False).apply(journey)


In [12]:
print("\nLast-touch channel (just before purchase) for converters:")
last_touch = converting['path'].apply(lambda p: p[-1]).value_counts(normalize=True)*100
print(last_touch.round(2))


Last-touch channel (just before purchase) for converters:
path
Google Search      48.40
Marketplace        18.62
Instagram          15.88
Influencer Blog     9.11
Youtube             7.98
Name: proportion, dtype: float64


## Step 4 — How often does each channel appear *anywhere* in a converting path?

This is the real tell. Google Search appears somewhere in 71.5% of converting
journeys — far more than the 48.4% it gets credit for under last-touch — confirming
that Search is involved in conversions well beyond just being the final click.
Meanwhile every other channel shows up in roughly half of converting journeys too,
meaning most conversions are genuinely multi-touch.

**Takeaway for modeling:** last-click attribution would dramatically undercount the
contribution of upper-funnel channels (Influencer Blog, Instagram, Marketplace,
Youtube). A multi-touch model (Markov removal-effect and/or Shapley value) is
necessary to credit channels fairly — this is the justification for the modeling
approach used in `Models.ipynb`.

In [14]:
# Critical check: in how many paths does Google Search appear ANYWHERE?
gs_anywhere = converting['path'].apply(lambda p: 'Google Search' in p).mean() * 100
print(f"\n% of converting journeys where Google Search appears ANYWHERE: {gs_anywhere:.2f}%")

for ch in df['channel'].unique():
    pct = converting['path'].apply(lambda p: ch in p).mean() * 100
    print(f"  {ch:<20} appears in {pct:.2f}% of converting journeys")


% of converting journeys where Google Search appears ANYWHERE: 71.52%
  Influencer Blog      appears in 59.59% of converting journeys
  Google Search        appears in 71.52% of converting journeys
  Marketplace          appears in 51.35% of converting journeys
  Instagram            appears in 52.38% of converting journeys
  Youtube              appears in 48.58% of converting journeys
